# GEAP Agent Evaluation — SDK-First Interactive Notebook

A **flat, teaching-oriented** walk of the whole **Quality Flywheel** with the **Vertex AI GenAI evals
SDK as a first-class citizen** — every phase calls `client.evals.*` and `vertexai.types.*` **inline**,
with no wrapper functions. It covers Google's
[Optimize → Evaluation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/agent-evaluation) docs end to end.

**L1 SDK vs custom.** We stick to the L1 SDK wherever possible. A few things are **not** eval-SDK
features and are called out with 🔧 — inference (a workaround for an SDK bug on the pinned version),
resilience/env-simulation, reading historical traces from BigQuery, online-monitor setup, the ADK GEPA
optimizer, and the Cloud Monitoring alert policy. Those reuse the repo's modules and are clearly labeled.

> Pinned to `google-cloud-aiplatform 1.162`. Where an L1 call is currently broken on that version the
> cell shows the documented call, the reason, and the minimal workaround. A headless version of the same
> flywheel runs via `uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID`.

## Setup

In [1]:
import os
# Run from the repo root so `from src...` imports and relative fixture paths resolve.
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

# --- Vertex AI GenAI evals SDK: the L1 entry points used throughout ---
import vertexai
from vertexai import Client, types, agent_engines
from google.genai import types as g_types
from src.config import GCP_PROJECT_ID, GCP_REGION, AGENT_ENGINE_ID

vertexai.init(project=GCP_PROJECT_ID, location=GCP_REGION)
client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)      # -> client.evals.* is the eval SDK

AGENT_RESOURCE = f"projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}/reasoningEngines/{AGENT_ENGINE_ID}"
print("client.evals ready:", hasattr(client, "evals"), "| agent:", AGENT_RESOURCE)

/tmp/ipykernel_3174447/2323432565.py:15: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)      # -> client.evals.* is the eval SDK


00:36:09 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


00:36:09 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


client.evals ready: True | agent: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024


### 🔧 One acknowledged helper: running inference

The documented L1 way to get responses is `client.evals.run_inference(agent=...)`. On
`aiplatform 1.162` it parses the Agent Engine's streamed events into the `AgentData` model
(`extra="forbid"`) and raises *"Failed to parse agent run response … to agent data: 'text'"*. So we
run inference ourselves via the Agent Engine `stream_query` API and hand the text to the SDK as
`EvalCase.responses`. **Everything else below is pure L1 SDK** (`client.evals.evaluate` + `types.*`).

In [2]:
# 🔧 CUSTOM (inference workaround) + tiny reporting helper. Scoring stays 100% L1 SDK.
_engine = agent_engines.get(AGENT_RESOURCE)

def agent_answer(prompt: str, user_id: str = "sdk-demo") -> str:
    """Query the deployed Agent Engine and return its final text (replaces run_inference on 1.162)."""
    texts = []
    for event in _engine.stream_query(message=prompt, user_id=user_id):
        for part in ((event.get("content") or {}).get("parts") or []):
            if part.get("text"):
                texts.append(part["text"])
    return "\n".join(texts).strip() or "(no response)"

def eval_case(prompt: str, answer: str, reference: str | None = None) -> types.EvalCase:
    """Wrap a prompt + response as an SDK EvalCase (extra='allow'; scored by client.evals.evaluate)."""
    kw = dict(
        prompt=g_types.Content(parts=[g_types.Part.from_text(text=prompt)], role="user"),
        responses=[types.ResponseCandidate(
            response=g_types.Content(parts=[g_types.Part.from_text(text=answer)], role="model"))],
    )
    if reference:
        kw["reference"] = g_types.Content(parts=[g_types.Part.from_text(text=reference)], role="model")
    return types.EvalCase(**kw)

def show_summary(result, threshold: float = 3.0):
    """🔧 reporting: SDK returns 0-1 scores; rescale x5 to the repo's 1-5 pass>=3 convention."""
    for m in (result.summary_metrics or []):
        mean = m.mean_score or 0.0
        score = mean * 5 if mean <= 1.0 else mean
        flag = "PASS" if score >= threshold else "FAIL"
        print(f"  {m.metric_name:34s} {score:4.2f}/5  [{flag}]  (errors={m.num_cases_error}/{m.num_cases_total})")

## Phase 1 — Design: the metric types (manage-metrics)
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

Three SDK metric types: **predefined rubric** (`types.RubricMetric.*`), **custom LLM-as-judge**
(`types.LLMMetric` + `types.MetricPromptBuilder`), and **custom deterministic code**
(`types.CodeExecutionMetric`). Define once, reuse — and optionally register in the Metric Registry
with `client.evals.create_evaluation_metric(...)`.

In [3]:
# 1) Predefined rubric metrics (Google-managed autoraters) — reference-free
prebuilt = [
    types.RubricMetric.FINAL_RESPONSE_QUALITY,
    types.RubricMetric.INSTRUCTION_FOLLOWING,
    types.RubricMetric.GENERAL_QUALITY,
]

# 2) Custom LLM-as-judge metric (natural-language rubric) — types.LLMMetric
policy_compliance = types.LLMMetric(
    name="policy_compliance",
    prompt_template=types.MetricPromptBuilder(
        instruction="Rate the agent's corporate expense-policy compliance.",
        criteria={"compliance": "Does the response correctly apply corporate expense limits and guide the user?"},
        rating_scores={"5": "proactive + correct", "4": "correct", "3": "applied, no guidance",
                       "2": "incorrect", "1": "ignores policy"},
    ),
)

# 3) Custom deterministic code metric — types.CodeExecutionMetric (server runs `evaluate(instance)->float`)
policy_limit_code = types.CodeExecutionMetric(
    name="policy_limit_exact",
    custom_function="""
def evaluate(instance: dict) -> float:
    text = str((instance or {}).get("response") or "").lower()
    limits = {"meal": 75, "meals": 75, "transport": 200, "lodging": 400, "supplies": 100, "entertainment": 150}
    hit = [c for c in limits if c in text]
    if not hit:
        return 0.5
    return 1.0 if any(str(limits[c]) in text for c in hit) else 0.0
""",
)

print("predefined rubric :", ["FINAL_RESPONSE_QUALITY", "INSTRUCTION_FOLLOWING", "GENERAL_QUALITY"])
print("custom LLM judge  :", policy_compliance.name, "(types.LLMMetric)")
print("custom code metric:", policy_limit_code.name, "(types.CodeExecutionMetric)")

# Register once, reuse across runs/monitors (writes to your project) — uncomment to run:
# print(client.evals.create_evaluation_metric(metric=policy_compliance))

predefined rubric : ['FINAL_RESPONSE_QUALITY', 'INSTRUCTION_FOLLOWING', 'GENERAL_QUALITY']
custom LLM judge  : policy_compliance (types.LLMMetric)
custom code metric: policy_limit_exact (types.CodeExecutionMetric)


## Phase 2a — Rapid evaluation (evaluate-agents)
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

Build an `EvaluationDataset`, score it with `client.evals.evaluate`, and render the SDK's interactive
table with `result.show()` (the [view-results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results) feature).

In [4]:
prompts = [
    "Find flights from SFO to JFK on June 15",
    "Search hotels in New York under $300",
    "Submit a $500 entertainment expense for user EMP001",
    "Check if a $50 meal expense is within policy",
    "Book flight FL001 for Jane Doe",
]
dataset = types.EvaluationDataset(eval_cases=[eval_case(p, agent_answer(p)) for p in prompts])

# --- L1 SDK scoring ---
result = client.evals.evaluate(dataset=dataset, metrics=prebuilt)
result.show()            # interactive aggregate + per-case tables
show_summary(result)

Computing Metrics for Evaluation Dataset:   0%|          | 0/15 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:   7%|▋         | 1/15 [00:11<02:46, 11.89s/it]

Computing Metrics for Evaluation Dataset:  13%|█▎        | 2/15 [00:13<01:19,  6.11s/it]

Computing Metrics for Evaluation Dataset:  20%|██        | 3/15 [00:14<00:45,  3.78s/it]

Computing Metrics for Evaluation Dataset:  27%|██▋       | 4/15 [00:15<00:25,  2.35s/it]

...

Computing Metrics for Evaluation Dataset:  87%|████████▋ | 13/15 [00:26<00:03,  1.93s/it]

Computing Metrics for Evaluation Dataset:  93%|█████████▎| 14/15 [00:28<00:01,  1.74s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 15/15 [00:38<00:00,  4.43s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 15/15 [00:38<00:00,  2.60s/it]

  final_response_quality_v1          3.00/5  [PASS]  (errors=0/5)
  instruction_following_v1           2.92/5  [FAIL]  (errors=0/5)
  general_quality_v1                 3.64/5  [PASS]  (errors=0/5)


## Phase 2b — Test-case / regression batch (evaluate-agents)
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

The same `client.evals.evaluate` call over a curated regression suite (a subset here for speed). The
test cases are project **data**; scoring is L1 SDK.

In [5]:
from src.eval.agent_eval_configs import get_eval_cases   # project test-case data (not SDK)

regression = get_eval_cases("coordinator_agent")[:6]
cases = [eval_case(c["prompt"], agent_answer(c["prompt"]), reference=c.get("reference")) for c in regression]
result = client.evals.evaluate(dataset=types.EvaluationDataset(eval_cases=cases), metrics=prebuilt)
show_summary(result)

Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


...

Computing Metrics for Evaluation Dataset:  89%|████████▉ | 16/18 [00:31<00:04,  2.28s/it]

Computing Metrics for Evaluation Dataset:  94%|█████████▍| 17/18 [00:32<00:01,  1.84s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 18/18 [00:36<00:00,  2.57s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 18/18 [00:36<00:00,  2.02s/it]

  final_response_quality_v1          2.78/5  [FAIL]  (errors=0/6)
  instruction_following_v1           3.50/5  [PASS]  (errors=0/6)
  general_quality_v1                 3.01/5  [PASS]  (errors=0/6)


## Phase 2c — Simulated scenario evaluation (evaluate-simulated)
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

`client.evals.generate_conversation_scenarios` synthesizes scenarios (a starting prompt + a hidden
conversation plan) grounded by an `environment_context`. We then run each scenario's opening turn and
score it. (Multi-turn `run_inference` is unavailable on 1.162, so we score the opening turn.)

In [6]:
from src.eval.agent_eval_configs import build_agent_info

agent_info = build_agent_info("coordinator_agent")

# --- L1 SDK: synthesize scenarios ---
scenarios = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 2,
        "generation_instruction": "Generate diverse corporate travel + expense conversation scenarios.",
        "environment_context": "Employee EMP001. Meal limit $75, lodging $400. Flights FL001/FL002; hotels HT001/HT002.",
    },
    allow_cross_region_model=True,
)

cases = []
for sc in (scenarios.eval_cases or []):
    opening = (getattr(getattr(sc, "user_scenario", None), "starting_prompt", "") or "").strip()
    if opening:
        cases.append(eval_case(opening, agent_answer(opening)))

result = client.evals.evaluate(dataset=types.EvaluationDataset(eval_cases=cases), metrics=prebuilt)
show_summary(result)

Computing Metrics for Evaluation Dataset:   0%|          | 0/6 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:  17%|█▋        | 1/6 [00:18<01:33, 18.77s/it]

Computing Metrics for Evaluation Dataset:  33%|███▎      | 2/6 [00:18<00:31,  7.86s/it]

Computing Metrics for Evaluation Dataset:  50%|█████     | 3/6 [00:19<00:13,  4.57s/it]

Computing Metrics for Evaluation Dataset:  67%|██████▋   | 4/6 [00:20<00:06,  3.09s/it]

Computing Metrics for Evaluation Dataset:  83%|████████▎ | 5/6 [00:30<00:05,  5.41s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:51<00:00, 10.87s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:51<00:00,  8.58s/it]

  final_response_quality_v1          3.33/5  [PASS]  (errors=0/2)
  instruction_following_v1           2.08/5  [FAIL]  (errors=0/2)
  general_quality_v1                 2.05/5  [FAIL]  (errors=0/2)


## Phase 2d — Environment simulation / resilience (evaluate-simulated) — 🔧 custom
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

Injecting mocked data + tool failures is **not** an eval-SDK feature — it wraps the agent's MCP tools.
Shown for completeness; the resulting traces would feed the multi-turn autoraters.

In [7]:
# 🔧 CUSTOM: mock tools + inject an HTTP 503 every 3rd call to test recovery.
from src.eval.env_simulation import run_with_env_simulation
run_with_env_simulation(inject_errors=True)

Environment simulation demo — agent_name='travel_agent'
In production, replace each MCP toolset on the agent with an
intercepted version, then run the simulated-user eval:
    live_tools = {t.name: t for t in agent.tools}         # MCP tools
    wrapped    = wrap_tools(live_tools, mocks=..., error_every=3)
    # ...attach `wrapped` to the agent, then:
    run_simulated_eval(agent_resource, agent_name=..., multi_turn=True)

Wrapped 3 tools: ['check_expense_policy', 'search_flights', 'search_hotels']
Error injection: every 3 call(s)
------------------------------------------------------------------------
  call 1: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline': 'Delta'}]
  call 2: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline'

{'agent_name': 'travel_agent',
 'agent_resource': None,
 'tools_wrapped': ['check_expense_policy', 'search_flights', 'search_hotels'],
 'total_calls': 6,
 'mock_hits': 5,
 'injected_errors': 1,
 'error_every': 3}

## Phase 2e — Offline evaluation over historical traces (evaluate-offline)
📖 [evaluate-offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline)

Score **already-recorded** prompt/response pairs — no new inference. **In production you read gen_ai
OTel traces from BigQuery**: the logging sink lands the gen_ai inference events in
`<dataset>.gen_ai_client_inference_operation_details_*`, with the OTel `gen_ai.*` attributes flattened
into the log's `labels` record. We read them here (🔧 custom — BigQuery isn't an eval-SDK feature) with
a fixture fallback, then score with the L1 SDK.

In [8]:
import json, pathlib
from google.cloud import bigquery
from src.config import BQ_EVAL_DATASET

# 🔧 CUSTOM: read already-recorded gen_ai OTel traces from BigQuery (gen_ai.* attrs live in labels.*).
def load_otel_traces_from_bigquery(limit: int = 20) -> list[dict]:
    bq = bigquery.Client(project=GCP_PROJECT_ID)
    sql = f"""
        SELECT labels.gen_ai_agent_name      AS agent,
               labels.gen_ai_input_messages  AS input_messages,
               labels.gen_ai_output_messages AS output_messages
        FROM `{GCP_PROJECT_ID}.{BQ_EVAL_DATASET}.gen_ai_client_inference_operation_details_*`
        WHERE labels.gen_ai_output_messages IS NOT NULL
        ORDER BY timestamp DESC
        LIMIT {int(limit)}
    """
    def flatten(blob):                                  # gen_ai messages -> plain text
        try:
            msgs = json.loads(blob)
        except Exception:
            return str(blob or "")
        parts = [str(p.get("content") or p.get("text") or "")
                 for m in (msgs or []) for p in (m.get("parts") or [])]
        return chr(10).join(x for x in parts if x).strip()
    recs = []
    for r in bq.query(sql).result():
        prompt, response = flatten(r["input_messages"]), flatten(r["output_messages"])
        if prompt and response:
            recs.append({"prompt": prompt, "response": response})
    return recs

try:
    traces = load_otel_traces_from_bigquery()
    source = f"BigQuery {BQ_EVAL_DATASET}.gen_ai_client_inference_operation_details_*"
except Exception as e:
    traces, source = [], f"BigQuery unavailable ({type(e).__name__})"
if not traces:                                          # graceful fallback so the demo always has data
    traces = [json.loads(x) for x in pathlib.Path("src/eval/sample_traces.jsonl").read_text().splitlines() if x.strip()]
    source += " -> bundled fixture"
print(f"Loaded {len(traces)} historical traces from: {source}")

# --- L1 SDK: score the recorded traces (no new inference) ---
cases = [eval_case(t["prompt"], t["response"]) for t in traces]
result = client.evals.evaluate(
    dataset=types.EvaluationDataset(eval_cases=cases),
    metrics=[types.RubricMetric.FINAL_RESPONSE_QUALITY,
             types.RubricMetric.HALLUCINATION,
             types.RubricMetric.SAFETY],
)
show_summary(result)

Loaded 10 historical traces from: BigQuery geap_workshop_logs.gen_ai_client_inference_operation_details_*


Computing Metrics for Evaluation Dataset:   0%|          | 0/30 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:   3%|▎         | 1/30 [00:03<01:31,  3.14s/it]

Computing Metrics for Evaluation Dataset:   7%|▋         | 2/30 [00:04<00:53,  1.92s/it]

Computing Metrics for Evaluation Dataset:  10%|█         | 3/30 [00:04<00:29,  1.10s/it]

...

Computing Metrics for Evaluation Dataset:  93%|█████████▎| 28/30 [00:37<00:06,  3.42s/it]

Computing Metrics for Evaluation Dataset:  97%|█████████▋| 29/30 [00:41<00:03,  3.37s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 30/30 [00:49<00:00,  4.81s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 30/30 [00:49<00:00,  1.64s/it]

  final_response_quality_v1          3.17/5  [PASS]  (errors=0/10)
  hallucination_v1                   5.00/5  [PASS]  (errors=0/10)
  safety_v1                          4.50/5  [PASS]  (errors=0/10)


### 🔧 How to actually set this up in production

The traces above come from **gen_ai OpenTelemetry** instrumentation on the deployed agent, routed to
BigQuery by a **Cloud Logging sink**. Two one-time steps:

**1. Instrument the agent** — set these env vars on the Agent Engine deployment (see
`src/config.py::OTEL_ENV_VARS`) so it emits gen_ai spans/events *with message content*:

```
GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY=true
OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental
OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=EVENT_ONLY
# + multimodal upload hook (image/audio/video content) — see OTEL_ENV_VARS in config
```

**2. Route the logs to BigQuery** — a logging sink lands them in
`gen_ai_client_inference_operation_details_*`. Run `bash scripts/setup_logging_sink.sh`, or:

```bash
bq mk --dataset "$GCP_PROJECT_ID:geap_workshop_logs"
gcloud logging sinks create geap-agent-traces \
    "bigquery.googleapis.com/projects/$GCP_PROJECT_ID/datasets/geap_workshop_logs" \
    --log-filter='resource.type="aiplatform.googleapis.com/AgentEngine"'
# then grant the sink's writerIdentity roles/bigquery.dataEditor on the dataset
```

Once traces are flowing, the cell above reads them directly. Console equivalent:
**Agent Platform → Agents → Evaluation → New evaluation → Traces/Sessions tab**.

## Phase 3 — Continuous evaluation with Online Monitors (evaluate-online) — 🔧 custom
📖 [evaluate-online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online)

Online Monitors asynchronously score **live production** traces on a ~10-minute loop and export scores
to Cloud Logging + Cloud Monitoring. They're created via the evaluation/Discovery Engine API (a setup
script), not `client.evals.evaluate`.

In [9]:
# 🔧 CUSTOM: create/verify online monitors via the setup script:
#     python -m src.eval.setup_online_evaluators create | verify
print("Online Monitors: async scoring of live traces (Query -> Evaluate -> Report),")
print("exported to Cloud Logging + Cloud Monitoring. See src/eval/setup_online_evaluators.py")

Online Monitors: async scoring of live traces (Query -> Evaluate -> Report),
exported to Cloud Logging + Cloud Monitoring. See src/eval/setup_online_evaluators.py


## Phase 4 — Optimize agent prompts (optimize-agent) — close the flywheel 🔁
📖 [optimize-agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent)

The documented L1 optimizer is `client.optimizer.optimize(targets=[...], benchmark=result, tests=dataset)`.
It isn't shipped in `aiplatform 1.162`, so we feature-detect it and otherwise point to the repo's ADK
GEPA fallback (🔧 custom), which re-tunes the coordinator's instruction against its eval set.

In [10]:
# --- L1 SDK (documented) ---
optimizer = getattr(client, "optimizer", None)
if optimizer is not None and callable(getattr(optimizer, "optimize", None)):
    opt = client.optimizer.optimize(targets=["system_prompt"], benchmark=result, tests=dataset)
    print("SDK optimizer result:", opt)
else:
    print("client.optimizer is not available in aiplatform 1.162 (the documented API is not shipped here).")
    print("🔧 Fallback — live GEPA optimization of the coordinator's instruction (requires reachable MCP):")
    print("   GEAP_RUN_GEPA=1 uv run python -m src.optimize.run_optimize src/agents/coordinator")
    print("   (or, from Python:  from src.eval.sdk_optimize import sdk_optimize;")
    print("                      sdk_optimize(client, run=True, max_metric_calls=30) )")

client.optimizer is not available in aiplatform 1.162 (the documented API is not shipped here).
🔧 Fallback — live GEPA optimization of the coordinator's instruction (requires reachable MCP):
   GEAP_RUN_GEPA=1 uv run python -m src.optimize.run_optimize src/agents/coordinator
   (or, from Python:  from src.eval.sdk_optimize import sdk_optimize;
                      sdk_optimize(client, run=True, max_metric_calls=30) )


## Quality-drift alerts (quality-alerts) — 🔧 custom
📖 [quality-alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts)

Export a Cloud Monitoring alert policy on the online-evaluator score metric, then apply it with gcloud.

In [11]:
# 🔧 CUSTOM: write the alert policy YAML (apply with `gcloud monitoring policies create --policy-from-file=...`)
from src.eval.quality_alerts import export_policy_yaml
print("policy file:", export_policy_yaml("src/eval/policies/quality_drift_policy.yaml"))

✓ Alert policy YAML written: src/eval/policies/quality_drift_policy.yaml
  Metric: aiplatform.googleapis.com/online_evaluator/scores (evaluation_metric_name=task_success) < 0.8
  Apply: gcloud monitoring policies create --policy-from-file=src/eval/policies/quality_drift_policy.yaml
policy file: src/eval/policies/quality_drift_policy.yaml


## Recap — L1 SDK vs. acknowledged custom code

**Pure L1 SDK (`client.evals.*` / `vertexai.types.*`):**
- `client.evals.evaluate(dataset, metrics)` — rapid, regression, simulated, and offline scoring
- `client.evals.generate_conversation_scenarios(...)` — synthetic multi-turn scenarios
- `client.evals.create_evaluation_metric(...)` — Metric Registry
- metrics: `types.RubricMetric.*`, `types.LLMMetric` + `types.MetricPromptBuilder`, `types.CodeExecutionMetric`
- data: `types.EvaluationDataset`, `types.EvalCase`, `types.ResponseCandidate`, `types.evals.AgentInfo`
- `result.show()` — view-results tables; documented `client.optimizer.optimize(...)` (feature-detected)

**🔧 Acknowledged custom (not eval-SDK, or SDK-bug workarounds):**
- inference via `agent_engines.stream_query` (works around the `run_inference` AgentData bug on 1.162)
- resilience / environment simulation (MCP tool mocking + fault injection)
- reading historical OTel traces from BigQuery
- online-monitor setup (`setup_online_evaluators.py`)
- ADK GEPA optimizer (`run_optimize.py`) — the fallback until `client.optimizer` ships
- Cloud Monitoring alert policy (gcloud)

Headless orchestrator + JSON report:
`uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID`